# YOLOE: Real-Time Seeing Anything — Code Companion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
[![GitHub](https://img.shields.io/badge/code-GitHub-blue.svg)](https://github.com/THU-MIG/yoloe)
[![arXiv](https://img.shields.io/badge/arXiv-2503.07465-b31b1b.svg)](https://arxiv.org/abs/2503.07465)

Hands-on, end-to-end tutorial accompanying the LearnOpenCV blog post **YOLOE: Real-Time Seeing Anything with Open-Vocabulary Detection and Segmentation**.

**What you will build**

1. **Environment setup** — a single `pip install`, platform-independent (Colab / Windows / macOS / Linux)
2. **Text-prompted detection and segmentation** on images — simple, multi-class, color-attribute, compositional, live prompt input, confidence sweeps, prompt-switching
3. **Visual-prompted detection and segmentation** on images — interactive box drawing, plus a portable hardcoded alternative
4. **Prompt-free detection and segmentation** — built-in 4,585-class open-world mode
5. **Raw data access** — boxes, masks, class IDs, confidences
6. **Text-prompted inference on video** — eight different scenes covering color attributes, size attributes, long-tail vocabulary, kitchen utensils, industrial scenes, aerial small objects, and more
7. **Visual-prompted inference on video** — draw a box once, track similar objects across the clip
8. **Object tracking** with persistent IDs across frames
9. **Test on different models:** YOLOE-26, YOLOE-11, and YOLOE-v8

Primary model: `yoloe-26l-seg.pt` (YOLOE on the newest YOLO26 backbone). Drop-in alternatives commented in every model-loading cell.


## 1. Environment Setup

Install the packages the notebook needs:

- **ultralytics** — YOLOE weights, inference, and the `YOLOE` Python class (auto-downloads checkpoints on first use)
- **supervision** — Roboflow's annotation and video I/O library
- **opencv-python** — image and video read/write
- **gdown** — downloads our test assets from a public Google Drive folder
- **jupyter_bbox_widget** *(optional)* — interactive bounding-box drawing for visual prompts (Colab/Jupyter only)


In [ ]:
!pip install -q ultralytics supervision opencv-python tqdm gdown
!pip install -q jupyter_bbox_widget


In [ ]:
# --- Silence Ultralytics' per-call startup banner so the output stays clean ---
import os, logging
os.environ["YOLO_VERBOSE"] = "False"
logging.getLogger("ultralytics").setLevel(logging.ERROR)

import pathlib
import urllib.request
import base64
import subprocess
import shutil
from base64 import b64encode

import cv2
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import HTML, display

import supervision as sv
from ultralytics import YOLOE
from ultralytics.models.yolo.yoloe import YOLOEVPSegPredictor

if torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"torch        : {torch.__version__}")
print(f"device       : {DEVICE}")
print(f"supervision  : {sv.__version__}")


## 2. Download Test Assets

We use a curated set of images and videos hosted on a public Google Drive folder.

**Images** (`data/images/`)
- `Animals.png` — pastoral farm scene with black, brown, white horses + cows + red barn
- `City_Life.png` — NYC street with bus, taxi, cyclist, pedestrians
- `home.png` — modern living room interior
- `Mango_Tree.png` — orchard close-up with ripe (yellow) and unripe (green) mangoes

**Videos** (`data/Videos/`) — see the per-section descriptions below for what each one demonstrates.


In [ ]:
DRIVE_FOLDER = "https://drive.google.com/drive/folders/14eGVJjUPDUb7oFu5U8G6L6xMLEaR058j?usp=sharing"
DATA_DIR = pathlib.Path("data")

if not DATA_DIR.exists() or not any(DATA_DIR.iterdir()):
    DATA_DIR.mkdir(exist_ok=True)
    import gdown
    gdown.download_folder(DRIVE_FOLDER, output=str(DATA_DIR), quiet=False, use_cookies=False)
else:
    print(f"{DATA_DIR} already populated; skipping download.")

IMG_DIR   = DATA_DIR / "images"
VIDEO_DIR = DATA_DIR / "Videos"

print("\nImages:")
for p in sorted(IMG_DIR.iterdir()):
    print(f"  {p.name:18s} {p.stat().st_size/1024/1024:5.1f} MB")

print("\nVideos:")
for p in sorted(VIDEO_DIR.iterdir()):
    print(f"  {p.name:20s} {p.stat().st_size/1024/1024:5.1f} MB")


## 3. Choosing a YOLOE Checkpoint

YOLOE available in multiple sizes (`n`, `s`, `m`, `l`, `x`) and in both prompt-capable (`*-seg.pt`) and prompt-free (`*-seg-pf.pt`) forms:

| Series | Example variant | Backbone | When to use |
|---|---|---|---|
| **YOLOE-26** (newest) | `yoloe-26l-seg.pt` | YOLO26 | Best accuracy/speed balance, latest improvements |
| YOLOE-11 | `yoloe-11l-seg.pt` | YOLO11 | Matches existing YOLO11 pipelines |
| YOLOE-v8 | `yoloe-v8l-seg.pt` | YOLOv8 | Used in earlier notebooks (Roboflow, THU-MIG) |


In [ ]:
# Primary model: YOLOE on YOLO26 backbone (newest)
MODEL_NAME    = "yoloe-26l-seg.pt"
PF_MODEL_NAME = "yoloe-26l-seg-pf.pt"

# --- Drop-in alternatives (uncomment to switch) ---
# MODEL_NAME, PF_MODEL_NAME = "yoloe-11l-seg.pt", "yoloe-11l-seg-pf.pt"   # YOLO11-based
# MODEL_NAME, PF_MODEL_NAME = "yoloe-v8l-seg.pt", "yoloe-v8l-seg-pf.pt"   # YOLOv8-based

model = YOLOE(MODEL_NAME)
if DEVICE != "cpu":
    model.to(DEVICE)
print(f"Loaded {MODEL_NAME} on {DEVICE}")


## 4. Helpers

Three small utilities used throughout the notebook:

- `annotate_and_show(...)` — overlays masks + boxes + labels and returns a PIL image for display
- `run_text_prompt(...)` — sets text classes and runs single-image inference
- `run_text_prompt_video(...)` — runs frame-by-frame text-prompt inference on a video and saves an annotated MP4 (used for every Section 9 subsection)
- `show_video(path)` — re-encodes the saved MP4 to browser-friendly H.264 with ffmpeg and embeds it inline
- `prompt_input(label, default)` — pauses for user-typed comma-separated class names, with a fallback default


In [ ]:
def annotate_and_show(image_bgr, detections, class_names=None, scale=1.0):
    """Draw masks, boxes, and labels on an image and return a PIL Image for notebook display."""
    annotated = image_bgr.copy()
    annotated = sv.MaskAnnotator(opacity=0.4).annotate(scene=annotated, detections=detections)
    annotated = sv.BoxAnnotator(thickness=2).annotate(scene=annotated, detections=detections)

    if class_names is not None and len(detections) > 0:
        labels = [
            f"{class_names[int(cid)]} {conf:.2f}"
            for cid, conf in zip(detections.class_id, detections.confidence)
        ]
        annotated = sv.LabelAnnotator(text_scale=0.5).annotate(
            scene=annotated, detections=detections, labels=labels
        )
    else:
        annotated = sv.LabelAnnotator(text_scale=0.5).annotate(scene=annotated, detections=detections)

    if scale != 1.0:
        h, w = annotated.shape[:2]
        annotated = cv2.resize(annotated, (int(w*scale), int(h*scale)))

    return Image.fromarray(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))


def run_text_prompt(model, image_path, names, conf=0.15):
    model.set_classes(names)
    image = cv2.imread(str(image_path))
    results = model.predict(image, conf=conf, verbose=False)
    detections = sv.Detections.from_ultralytics(results[0])
    return detections, image


def run_text_prompt_video(model, src_video, out_video, names, conf=0.15, desc=None):
    """Run YOLOE text-prompted inference frame-by-frame and write an annotated MP4.

    Returns (out_video_path, total_detections_across_all_frames).
    """
    model.set_classes(names)
    info        = sv.VideoInfo.from_video_path(str(src_video))
    frames_iter = sv.get_video_frames_generator(str(src_video))

    mask_anno  = sv.MaskAnnotator(opacity=0.4)
    box_anno   = sv.BoxAnnotator(thickness=2)
    label_anno = sv.LabelAnnotator(text_scale=0.5)

    total = 0
    desc = desc or pathlib.Path(src_video).name
    with sv.VideoSink(str(out_video), info) as sink:
        for frame in tqdm(frames_iter, total=info.total_frames, desc=desc):
            results = model.predict(frame, conf=conf, verbose=False)
            detections = sv.Detections.from_ultralytics(results[0])
            total += len(detections)
            annotated = frame.copy()
            annotated = mask_anno.annotate(scene=annotated, detections=detections)
            annotated = box_anno.annotate(scene=annotated, detections=detections)
            labels = [
                f"{names[int(c)]} {p:.2f}"
                for c, p in zip(detections.class_id, detections.confidence)
            ]
            annotated = label_anno.annotate(scene=annotated, detections=detections, labels=labels)
            sink.write_frame(annotated)
    return out_video, total


def show_video(path, width=720):
    """Play a video inline (re-encoded to H.264 + yuv420p for browser compatibility)."""
    src = str(path)
    out = src.rsplit(".", 1)[0] + "_h264.mp4"

    if shutil.which("ffmpeg"):
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-i", src,
             "-c:v", "libx264", "-preset", "ultrafast", "-pix_fmt", "yuv420p", out],
            check=True,
        )
        playback_path = out
    else:
        playback_path = src

    with open(playback_path, "rb") as f:
        b64 = b64encode(f.read()).decode("ascii")
    return HTML(f'''
    <video width="{width}" controls autoplay loop muted playsinline>
      <source src="data:video/mp4;base64,{b64}" type="video/mp4">
    </video>
    ''')


def prompt_input(label, default):
    """Ask for comma-separated class names; fall back to default if no input."""
    try:
        raw = input(f"{label} (comma-separated, e.g., '{', '.join(default)}'): ")
    except EOFError:
        raw = ""
    names = [n.strip() for n in raw.split(",") if n.strip()]
    return names if names else list(default)


# Shared video-loop annotators (referenced by Sections 10 and 11)
mask_anno  = sv.MaskAnnotator(opacity=0.4)
box_anno   = sv.BoxAnnotator(thickness=2)
label_anno = sv.LabelAnnotator(text_scale=0.5)


## 5. Text-Prompted Inference

Text prompts are the most intuitive way to use YOLOE. You pass a list of natural-language class names to `model.set_classes(names)`, and YOLOE detects any object matching those descriptions. CLIP-style text embeddings + the **RepRTA** module make the head vocabulary-agnostic — you can type any concept, not just COCO classes.

We walk through six examples, from simple to compositional.

### 5.1 Live prompt input — type your own classes

The cell below pauses and asks you to type the classes you want to detect. Try `horse`, then re-run with `horse, cow, barn, pond` to see how multi-class prompting works on the same image.

In [ ]:
IMG_PATH = IMG_DIR / "Animals.png"

NAMES = prompt_input("Classes for detection", default=["horse", "cow"])
print(f"Using prompts: {NAMES}\n")

detections, image = run_text_prompt(model, IMG_PATH, names=NAMES, conf=0.2)

print(f"-> {len(detections)} detection(s)")
annotate_and_show(image, detections, class_names=NAMES, scale=0.5)


In [ ]:
IMG_PATH = IMG_DIR / "Animals.png"

NAMES = prompt_input("Classes for detection", default=["horse", "cow"])
print(f"Using prompts: {NAMES}\n")

detections, image = run_text_prompt(model, IMG_PATH, names=NAMES, conf=0.2)

print(f"-> {len(detections)} detection(s)")
annotate_and_show(image, detections, class_names=NAMES, scale=0.5)


### 5.2 Multi-class long-tail prompts on a city scene

Busy NYC street: COCO-standard categories (person, bicycle, bus, traffic light) plus things COCO doesn't have (specifically yellow taxi, US flag). Adding more class names is essentially free.

In [ ]:
IMG_PATH = IMG_DIR / "City_Life.png"
NAMES = ["person", "bicycle", "bus", "traffic light", "car", "backpack"]

detections, image = run_text_prompt(model, IMG_PATH, names=NAMES, conf=0.2)

print(f"Prompts: {NAMES}\n-> {len(detections)} detection(s)")
annotate_and_show(image, detections, class_names=NAMES, scale=0.5)


### 5.3 Color-attribute prompts — what closed-set YOLO cannot do

Black + brown + white horses in the same frame. A COCO-trained YOLO would label all three as `"horse"`. YOLOE separates them by color via CLIP-style attribute matching.

In [ ]:
IMG_PATH = IMG_DIR / "Animals.png"
NAMES = ["white horse", "brown horse", "black horse"]

detections, image = run_text_prompt(model, IMG_PATH, names=NAMES, conf=0.15)

print(f"Prompts: {NAMES}\n-> {len(detections)} detection(s)")
annotate_and_show(image, detections, class_names=NAMES, scale=0.5)


**Takeaway:** color attributes work because CLIP was trained on millions of captioned images that frequently mention colors. Same pattern works for size (`"small"`, `"large"`), pattern (`"striped"`), or material (`"wooden"`).

### 5.4 Compositional prompts — Colors

Apple orchard with ripe (yellow-orange) and unripe (green) apples hanging on the same tree.

In [ ]:
IMG_PATH = IMG_DIR / "apples.png"

NAMES = prompt_input("Mango-tree prompts", default=["red apple", "green apple"])
print(f"Using prompts: {NAMES}\n")

detections, image = run_text_prompt(model, IMG_PATH, names=NAMES, conf=0.1)

print(f"-> {len(detections)} detection(s)")
annotate_and_show(image, detections, class_names=NAMES, scale=0.5)


### 5.5 Confidence threshold — how it changes what you see

Open-vocabulary detectors return *lower* confidence scores than closed-set YOLOs. A good range for YOLOE is **`conf=0.10` to `conf=0.25`**. Below we sweep three settings on the mango-tree image.

In [ ]:
import matplotlib.pyplot as plt

#IMG_PATH = IMG_DIR / "apples.png"
IMG_PATH =  "/content/apples.png"
NAMES = ["red apples", "green apples"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, conf in zip(axes, [0.05, 0.15]):
    detections, image = run_text_prompt(model, IMG_PATH, names=NAMES, conf=conf)
    img_pil = annotate_and_show(image, detections, class_names=NAMES, scale=0.35)
    ax.imshow(img_pil); ax.axis("off")
    ax.set_title(f"conf={conf}  ({len(detections)} dets)")

plt.tight_layout(); plt.show()


### 5.6 Switching prompts on the fly

`set_classes(...)` only updates prompt embeddings (not weights), so call it as many times as you want. No reload, no recompile.

In [ ]:
IMG_PATH = IMG_DIR / "Animals.png"

prompt_sets = [
    ["horse"],
    ["cow"],
    ["red barn", "white horse"],
]

for names in prompt_sets:
    detections, image = run_text_prompt(model, IMG_PATH, names=names, conf=0.15)
    print(f"\n>>> Prompts: {names}  ->  {len(detections)} detection(s)")
    display(annotate_and_show(image, detections, class_names=names, scale=0.4))


## 6. Visual-Prompted Inference

**The bounding box IS your prompt.** In text mode, you type `"sofa"`. In visual mode, you draw a box around one sofa or one framed picture — that box is your question. YOLOE then finds objects visually similar to the boxed region.

Visual prompts shine when an object is hard to describe in words (specific industrial parts, distinctive logos, particular animal markings). The Ultralytics API is a single call:

```python
model.predict(
    target_image,
    refer_image=reference_image,
    visual_prompts=dict(bboxes=..., cls=...),
    predictor=YOLOEVPSegPredictor,
)
```


### 6.1 Draw your own reference boxes (interactive)

Draw a box around **one potted plant**, **one framed picture on the wall**, and **the sofa**, pick the right label from the dropdown for each, and click **Submit**.

In [ ]:
def encode_image_for_widget(source):
    if isinstance(source, (str, pathlib.Path)):
        with open(source, "rb") as f:
            image_bytes = f.read()
    else:
        ok, buf = cv2.imencode(".jpg", source)
        if not ok:
            raise ValueError("Could not JPEG-encode image for widget.")
        image_bytes = buf.tobytes()
    b64 = base64.b64encode(image_bytes).decode("utf-8")
    return f"data:image/jpg;base64,{b64}"


try:
    from jupyter_bbox_widget import BBoxWidget
    WIDGET_AVAILABLE = True
except ImportError:
    WIDGET_AVAILABLE = False
    print("jupyter_bbox_widget not available; jump to Section 6.3 for hardcoded boxes.")

if WIDGET_AVAILABLE:
    try:
        from google.colab import output as _colab_output
        _colab_output.enable_custom_widget_manager()
    except ImportError:
        pass

    VP_NAMES = ["potted plant", "framed art", "sofa"]
    widget = BBoxWidget(classes=VP_NAMES)
    widget.image = encode_image_for_widget(IMG_DIR / "home.png")
    display(widget)


In [ ]:
# Rough coordinates for home.png — adjust to match what you see in the widget
DEFAULT_BOXES = [
    {"x":  450, "y":  120, "width":  450, "height": 1080, "label": "potted plant"},
    {"x": 1100, "y":  280, "width":  220, "height":  280, "label": "framed art"},
    {"x": 1430, "y":  580, "width":  500, "height":  450, "label": "sofa"},
]

drawn = widget.bboxes if (WIDGET_AVAILABLE and widget.bboxes) else DEFAULT_BOXES
VP_NAMES = ["potted plant", "framed art", "sofa"]

bboxes = np.array([
    [b["x"], b["y"], b["x"] + b["width"], b["y"] + b["height"]]
    for b in drawn
], dtype=np.float64)
cls = np.array([VP_NAMES.index(b["label"]) for b in drawn], dtype=np.int32)

print(f"Captured {len(drawn)} box(es):")
for b in drawn:
    print(f"  {b['label']:14s} x={b['x']:4d} y={b['y']:4d} w={b['width']:4d} h={b['height']:4d}")


### 6.2 Run inference with your drawn boxes

The model uses the regions you boxed as queries and returns every visually similar object in the image.

In [ ]:
IMG_PATH = IMG_DIR / "home.png"

vp_model = YOLOE(MODEL_NAME)
if DEVICE != "cpu":
    vp_model.to(DEVICE)

results = vp_model.predict(
    str(IMG_PATH),
    refer_image=str(IMG_PATH),
    visual_prompts=dict(bboxes=bboxes, cls=cls),
    predictor=YOLOEVPSegPredictor,
    conf=0.15,
    verbose=False,
)
detections = sv.Detections.from_ultralytics(results[0])

target_bgr = cv2.imread(str(IMG_PATH))
print(f"{len(detections)} detections using your drawn boxes ({VP_NAMES})")
annotate_and_show(target_bgr, detections, class_names=VP_NAMES, scale=0.5)


### 6.3 Reuse your boxes for batch processing or scripts

Once you've drawn boxes in the widget, you can save them and reuse the same prompt across any number of images without re-drawing. Useful when you want to ship a deployment script. The cell below prints the boxes you just drew in a copy-pasteable format.

In [ ]:
img = cv2.imread(str(IMG_DIR / "home.png"))
print(f"home.png is {img.shape[1]} x {img.shape[0]} (W x H)")

In [ ]:
# Print your widget output as a copy-pasteable Python literal
if WIDGET_AVAILABLE and widget.bboxes:
    print("# Save these and reuse them later:")
    print(f"FIXED_NAMES = {VP_NAMES}")
    print("FIXED_BBOXES = np.array([")
    for b in widget.bboxes:
        x1, y1 = b['x'], b['y']
        x2, y2 = x1 + b['width'], y1 + b['height']
        print(f"    [{x1:5d}, {y1:5d}, {x2:5d}, {y2:5d}],   # {b['label']}")
    print("], dtype=np.float64)")
    print(f"FIXED_CLS    = np.array({[VP_NAMES.index(b['label']) for b in widget.bboxes]}, dtype=np.int32)")
else:
    print("No widget output to save. Run Section 6.1 first.")

## 7. Prompt-Free Inference — Open-World Labeling

The `-pf` checkpoints behave like a closed-set YOLO but over a built-in vocabulary of **4,585 categories** (LVIS + Objects365). The **LRPC** head contrasts every region proposal against this vocabulary and picks the best match — no prompt required.

Ideal for: **exploratory labeling** ("what's in this image?"), **data mining** (find unusual objects in a large set), and **bootstrapping annotations**.

In [ ]:
pf_model = YOLOE(PF_MODEL_NAME)
if DEVICE != "cpu":
    pf_model.to(DEVICE)

image = cv2.imread(str(IMG_DIR / "home.png"))
results = pf_model.predict(image, conf=0.25, verbose=False)
detections = sv.Detections.from_ultralytics(results[0])

unique_classes = sorted({pf_model.names[int(c)] for c in detections.class_id})
print(f"{len(detections)} detections across {len(unique_classes)} classes:")
print("  " + ", ".join(unique_classes))

class_names_lookup = [pf_model.names[i] for i in range(len(pf_model.names))]
annotate_and_show(image, detections, class_names=class_names_lookup, scale=0.5)


###  Peek at the 4,585-class vocabulary

In [ ]:
names = pf_model.names
print(f"Total classes : {len(names)}")
print(f"First 20      : {[names[i] for i in range(20)]}")
print(f"Middle 10     : {[names[i] for i in range(2000, 2010)]}")
print(f"Last 10       : {[names[i] for i in range(len(names)-10, len(names))]}")


## 8. Accessing Raw Detections and Masks

YOLOE exposes every detection as structured data: bounding boxes, class IDs, confidences, and per-instance binary masks. The `supervision.Detections` object gives a clean numpy-first interface.

### 8.1 Rae Detections

In [ ]:
image = cv2.imread(str(IMG_DIR / "home.png"))
results = pf_model.predict(image, conf=0.25, verbose=False)
detections = sv.Detections.from_ultralytics(results[0])

print(f"Number of detections : {len(detections)}")
print(f"Boxes  (N, 4) xyxy    : {detections.xyxy.shape}")
print(f"Class IDs    (N,)     : {detections.class_id.shape}")
print(f"Confidences  (N,)     : {detections.confidence.shape}")
if detections.mask is not None:
    print(f"Masks   (N, H, W)     : {detections.mask.shape}  (boolean)\n")

for i, (xyxy, cid, conf) in enumerate(zip(detections.xyxy, detections.class_id, detections.confidence)):
    name = pf_model.names[int(cid)]
    print(f"  {i:2d}  {name:20s}  conf={conf:.3f}  box={xyxy.round(1).tolist()}")


### 8.2 Extract a per-object mask as PNG

In [ ]:
if detections.mask is not None and len(detections) > 0:
    areas = detections.mask.sum(axis=(1, 2))
    idx = int(np.argmax(areas))
    mask = detections.mask[idx].astype(np.uint8) * 255
    cv2.imwrite("largest_mask.png", mask)

    cls_name = pf_model.names[int(detections.class_id[idx])]
    print(f"Saved largest mask (class='{cls_name}', area={areas[idx]} px) to largest_mask.png")
    display(Image.fromarray(mask).resize((mask.shape[1]//2, mask.shape[0]//2)))
else:
    print("No masks in this result (are you using a -seg variant?)")


## 9. Text-Prompted Inference on Video

Video inference is frame-by-frame. Because `set_classes(...)` caches prompt embeddings once, per-frame cost is dominated by the convolutional backbone — roughly the same as closed-set YOLO.

Each subsection below uses a different test video to highlight a specific YOLOE capability — color attributes, size attributes, long-tail vocabulary, compositional prompts, industrial scenes, indoor settings, and aerial small-object detection. They all share the same `run_text_prompt_video(...)` helper from Section 4, so each cell stays compact.

### 9.1 Multi-color suitcases — color-attribute detection (live input)

Airport baggage carousel with red, green, black, navy, tan, and silver suitcases. Closed-set YOLO would lump all of these into a single `"suitcase"` class. YOLOE separates them by color. Try the default prompts, or type your own combination of color + suitcase.


In [ ]:
show_video(VIDEO_DIR / "Suitcases.mp4", width=480)

In [ ]:
SRC_VIDEO = VIDEO_DIR / "Suitcases.mp4"
OUT_VIDEO = "output_suitcases.mp4"

NAMES = prompt_input("Prompts for Suitcases.mp4", default=['red suitcase', 'white suitcase', 'black suitcase', 'blue suitcase'])
print(f"Using prompts: {NAMES}\n")

vm = YOLOE(MODEL_NAME)
if DEVICE != "cpu":
    vm.to(DEVICE)

run_text_prompt_video(vm, SRC_VIDEO, OUT_VIDEO, NAMES, conf=0.15)
print(f"\nSaved {OUT_VIDEO}")


In [ ]:
show_video(OUT_VIDEO, width=720)


**Why this matters:** the same color-attribute trick works for any object class. Try `"red car"` + `"blue car"` on a parking lot, or `"yellow shirt person"` + `"black shirt person"` on a crowd.

### 9.2 Elephants — size-attribute detection (adult vs baby)

An adult elephant with juvenile and baby elephants on grass. Size attributes work just like color attributes — `"adult elephant"` and `"baby elephant"` get separated by the model, even though COCO has only the single class `"elephant"`.


In [ ]:
SRC_VIDEO = VIDEO_DIR / "Elephants.mp4"
OUT_VIDEO = "output_elephants.mp4"
NAMES = ['adult elephant', 'baby elephant']

vm = YOLOE(MODEL_NAME)
if DEVICE != "cpu":
    vm.to(DEVICE)

run_text_prompt_video(vm, SRC_VIDEO, OUT_VIDEO, NAMES, conf=0.15)
print(f"\nSaved {OUT_VIDEO}")


In [ ]:
show_video(OUT_VIDEO, width=720)


### 9.3 Kitchen utensils and ingredients (live input)

Cooking scene with whisk, eggs, glass jars, cherry tomatoes, banana, basket, and cutting board. Try `whisk, egg, tomato, banana, jar` — or any combination of kitchen items you want.


In [ ]:
SRC_VIDEO = VIDEO_DIR / "Kitchen.mp4"
OUT_VIDEO = "output_kitchen.mp4"

NAMES = prompt_input("Prompts for Kitchen.mp4", default=['whisk', 'egg', 'tomato', 'banana', 'jar'])
print(f"Using prompts: {NAMES}\n")

vm = YOLOE(MODEL_NAME)
if DEVICE != "cpu":
    vm.to(DEVICE)

run_text_prompt_video(vm, SRC_VIDEO, OUT_VIDEO, NAMES, conf=0.15)
print(f"\nSaved {OUT_VIDEO}")


In [ ]:
show_video(OUT_VIDEO, width=720)


### 9.4 Banana market — single-class color demo

Vendor with hanging banana bunches against a colorful market backdrop. Yellow bananas everywhere — a good test that the color attribute doesn't get confused with similarly-colored objects (the lights, the painted signs).


In [ ]:
SRC_VIDEO = VIDEO_DIR / "Market.mp4"
OUT_VIDEO = "output_market.mp4"
NAMES = ['ripe banana', 'yellow banana']

vm = YOLOE(MODEL_NAME)
if DEVICE != "cpu":
    vm.to(DEVICE)

run_text_prompt_video(vm, SRC_VIDEO, OUT_VIDEO, NAMES, conf=0.15)
print(f"\nSaved {OUT_VIDEO}")


In [ ]:
show_video(OUT_VIDEO, width=720)


### 9.5 Warehouse forklift — long-tail industrial scene

Toyota forklift moving through a warehouse with shelves of boxes, pallets, and equipment. Industrial scenes are a domain where COCO's vocabulary is thin — `"forklift"`, `"cardboard box"`, and `"warehouse shelf"` are all out of distribution for closed-set models, but YOLOE handles them via text prompts.


In [ ]:
SRC_VIDEO = VIDEO_DIR / "Fork_Kifter.mp4"
OUT_VIDEO = "output_fork_kifter.mp4"
NAMES = ['forklift', 'cardboard box', 'warehouse shelf']

vm = YOLOE(MODEL_NAME)
if DEVICE != "cpu":
    vm.to(DEVICE)

run_text_prompt_video(vm, SRC_VIDEO, OUT_VIDEO, NAMES, conf=0.15)
print(f"\nSaved {OUT_VIDEO}")


In [ ]:
show_video(OUT_VIDEO, width=720)


### 9.6 Office desk top-down (live input)

Top-down view of a desk with a hand reaching to a laptop trackpad: scissors, tape dispenser, pencil case, tea mug, roses in a vase, notebook. A great everyday scene for trying any office prompt you can think of.


In [ ]:
SRC_VIDEO = VIDEO_DIR / "Office_Desk_2.mp4"
OUT_VIDEO = "output_office_desk_2.mp4"

NAMES = prompt_input("Prompts for Office_Desk_2.mp4", default=['scissors', 'tape dispenser', 'pencil case', 'tea mug', 'roses', 'laptop'])
print(f"Using prompts: {NAMES}\n")

vm = YOLOE(MODEL_NAME)
if DEVICE != "cpu":
    vm.to(DEVICE)

run_text_prompt_video(vm, SRC_VIDEO, OUT_VIDEO, NAMES, conf=0.15)
print(f"\nSaved {OUT_VIDEO}")


In [ ]:
show_video(OUT_VIDEO, width=720)


### 9.7 Office desk static — gadgets and stationery

Top-down still-life with a closed laptop, eyeglasses, plant, takeout coffee cup, DSLR camera, and notebook. Different angle and lighting from 9.6 — useful for cross-checking how robust your prompts are.


In [ ]:
SRC_VIDEO = VIDEO_DIR / "Office_Desk.mp4"
OUT_VIDEO = "output_office_desk.mp4"
NAMES = ['laptop', 'eyeglasses', 'camera', 'potted plant', 'coffee cup']

vm = YOLOE(MODEL_NAME)
if DEVICE != "cpu":
    vm.to(DEVICE)

run_text_prompt_video(vm, SRC_VIDEO, OUT_VIDEO, NAMES, conf=0.15)
print(f"\nSaved {OUT_VIDEO}")


In [ ]:
show_video(OUT_VIDEO, width=720)


### 9.8 Aerial village — small-object stress test

Drone shot over a tropical village: tin-roofed houses, palm trees, rice paddies, dirt roads. Aerial scenes have many small objects per frame, which is harder for any detector. Try lowering `conf` if recall is low.


In [ ]:
SRC_VIDEO = VIDEO_DIR / "village_Home.mp4"
OUT_VIDEO = "output_village_home.mp4"
NAMES = ['white roof house', 'palm tree', 'rice paddy']

vm = YOLOE(MODEL_NAME)
if DEVICE != "cpu":
    vm.to(DEVICE)

run_text_prompt_video(vm, SRC_VIDEO, OUT_VIDEO, NAMES, conf=0.1)
print(f"\nSaved {OUT_VIDEO}")


In [ ]:
show_video(OUT_VIDEO, width=720)


**Tip — for aerial videos:** consider downscaling to 720p before inference if your source is 4K (saves ~3× compute), and try `imgsz=1280` in `predict(...)` for better small-object recall on Large/X variants.

### 9.9 Running horses — motion-attribute detection

Aerial drone footage of a horse herd galloping across red-dirt terrain. Motion-attribute prompts (`"running horse"`, `"galloping horse"`) work because CLIP captures action verbs from caption data.


In [ ]:
SRC_VIDEO = VIDEO_DIR / "horses.mp4"
OUT_VIDEO = "output_horses.mp4"
NAMES = ['running horse', 'horse herd']

vm = YOLOE(MODEL_NAME)
if DEVICE != "cpu":
    vm.to(DEVICE)

run_text_prompt_video(vm, SRC_VIDEO, OUT_VIDEO, NAMES, conf=0.15)
print(f"\nSaved {OUT_VIDEO}")


In [ ]:
show_video(OUT_VIDEO, width=720)


## 10. Visual-Prompted Inference on Video

Same single-call API as Section 6, applied per frame. We use **`Food.mp4`** — a bakery conveyor with cake, croissants, and pastries — and box one croissant in the first frame. The model should track *just the croissants* across the rest of the clip, ignoring the cake and other pastries.

In [ ]:
SRC_VIDEO = VIDEO_DIR / "Food.mp4"
OUT_VIDEO = "output_food.mp4"

NAMES = prompt_input("Food prompts", default=["croissant", "cake", "burger", "pastry"])
print(f"Using prompts: {NAMES}\n")

vm = YOLOE(MODEL_NAME)
if DEVICE != "cpu":
    vm.to(DEVICE)

run_text_prompt_video(vm, SRC_VIDEO, OUT_VIDEO, NAMES, conf=0.15)
print(f"\nSaved {OUT_VIDEO}")

In [ ]:
show_video(OUT_VIDEO, width=720)


## 11. Bonus — Object Tracking With Persistent IDs

YOLOE integrates with Ultralytics' built-in trackers (ByteTrack and BoT-SORT) via `model.track(...)`. Each detection gets a `tracker_id` that stays consistent across frames — required for counting, trajectory estimation, and behavior analysis.

###  Airport — people walking with luggage

Terminal scene with people moving past departure gate signs. IDs should remain stable as people cross the frame.

In [ ]:
SRC_VIDEO = VIDEO_DIR / "airport.mp4"
OUT_VIDEO = "output_airport_tracked.mp4"
NAMES     = ["person", "rolling suitcase"]

track_model = YOLOE(MODEL_NAME)
if DEVICE != "cpu":
    track_model.to(DEVICE)
track_model.set_classes(NAMES)

info = sv.VideoInfo.from_video_path(str(SRC_VIDEO))

results_stream = track_model.track(
    source=str(SRC_VIDEO),
    conf=0.15,
    persist=True,
    tracker="bytetrack.yaml",
    stream=True,
    verbose=False,
)

trace_anno = sv.TraceAnnotator(trace_length=30, thickness=2)

with sv.VideoSink(OUT_VIDEO, info) as sink:
    for r in tqdm(results_stream, total=info.total_frames, desc="Tracking airport"):
        detections = sv.Detections.from_ultralytics(r)
        annotated  = r.orig_img.copy()
        annotated  = mask_anno.annotate(scene=annotated, detections=detections)
        annotated  = box_anno.annotate(scene=annotated, detections=detections)
        if detections.tracker_id is not None and len(detections) > 0:
            annotated = trace_anno.annotate(scene=annotated, detections=detections)
            labels = [
                f"#{int(tid)} {NAMES[int(cid)]} {conf:.2f}"
                for cid, conf, tid in zip(detections.class_id, detections.confidence, detections.tracker_id)
            ]
        else:
            labels = [f"{NAMES[int(c)]} {p:.2f}" for c, p in zip(detections.class_id, detections.confidence)]
        annotated = label_anno.annotate(scene=annotated, detections=detections, labels=labels)
        sink.write_frame(annotated)

print(f"Saved {OUT_VIDEO}")


In [ ]:
show_video(OUT_VIDEO, width=720)


## 12. Bonus — YOLOE-26 vs YOLOE-11 vs YOLOE-v8

YOLOE's three backbone series share the same open-vocabulary head but use different feature extractors. The cell below runs the same prompts on the same image across all three Large variants and prints detection count + inference latency.

In [ ]:
import time
import matplotlib.pyplot as plt

COMPARE_IMG = IMG_DIR / "Animals.png"
NAMES       = ["white horse", "brown horse", "black horse", "cow"]

variants = [
    ("yoloe-26l-seg.pt", "YOLOE-26 L"),
    ("yoloe-11l-seg.pt", "YOLOE-11 L"),
    ("yoloe-v8l-seg.pt", "YOLOE-v8 L"),
]

fig, axes = plt.subplots(1, len(variants), figsize=(18, 6))
image_bgr = cv2.imread(str(COMPARE_IMG))

for ax, (weights, label) in zip(axes, variants):
    m = YOLOE(weights)
    if DEVICE != "cpu":
        m.to(DEVICE)
    m.set_classes(NAMES)
    _ = m.predict(image_bgr, conf=0.15, verbose=False)  # warm-up

    t0 = time.time()
    results = m.predict(image_bgr, conf=0.15, verbose=False)
    dt_ms = (time.time() - t0) * 1000

    detections = sv.Detections.from_ultralytics(results[0])
    preview = annotate_and_show(image_bgr, detections, class_names=NAMES, scale=0.4)
    ax.imshow(preview); ax.axis("off")
    ax.set_title(f"{label}\n{len(detections)} dets  |  {dt_ms:.1f} ms")

plt.tight_layout(); plt.show()


## 13. Wrapping Up

You now have a complete YOLOE pipeline covering text, visual, and prompt-free inference on images and videos, with the newest YOLOE-26 backbone as the primary model and YOLOE-11 / YOLOE-v8 available as one-line swaps. You've seen ten different video scenes — color attributes, size attributes, long-tail vocabulary, compositional prompts, industrial scenes, indoor settings, and aerial small-object detection — plus tracking with persistent IDs and a three-way model comparison.

**Ideas for extending the notebook**

- **Custom fine-tuning.** Use `YOLOEPETrainer` (detection) or `YOLOEPESegTrainer` (segmentation) to adapt YOLOE to a domain-specific dataset.
- **ONNX / TensorRT export.** Call `set_classes(...)` to freeze your chosen vocabulary, then `model.export(format="onnx")`. Text-prompt embeddings become part of the exported graph.
- **Analytics pipelines.** Feed `detections` into `supervision.ByteTrack`, `LineZone`, `PolygonZone`, or `DetectionsDataset` for counters, heatmaps, dwell-time analysis.
- **Batched inference.** Pass a list of image paths to `model.predict([...])` for higher throughput than frame-by-frame loops.

**References**

- [Ultralytics YOLOE Documentation](https://docs.ultralytics.com/models/yoloe/)
- [YOLOE: Real-Time Seeing Anything (arXiv 2503.07465)](https://arxiv.org/abs/2503.07465)
- [Official YOLOE repository (THU-MIG)](https://github.com/THU-MIG/yoloe)
- [Roboflow `supervision` library](https://supervision.roboflow.com/)
- [LearnOpenCV Blog](https://learnopencv.com/)
